## Adaptación de Encoder Visual OpenCLIP para Portadas de Libros

Este notebook implementa la adaptación de un encoder visual OpenCLIP al dominio de portadas de libros utilizando el dataset BookCover30. El objetivo es mejorar la representación visual de portadas para un sistema de recomendación multimodal, manteniendo la compatibilidad con un catálogo existente.

### 1. Configuración Inicial y Dependencias

In [ ]:
# Instalación de librerías necesarias.
# open_clip: Para trabajar con los modelos CLIP.
# timm: PyTorch Image Models, una colección de modelos, capas y utilidades para visión por computadora.
# tqdm: Para barras de progreso durante el entrenamiento.
!pip install open_clip_torch timm tqdm opencv-python-headless

# Montar Google Drive para acceder a los datos y guardar los modelos.
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
import numpy as np
from PIL import Image
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import open_clip
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

# Configuración para reproducibilidad
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Determinar el dispositivo a usar (GPU si está disponible, de lo contrario CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

### 2. Verificar Estructura del BookCover30

In [ ]:
# --- 2.1 Cargar Rutas de Imágenes y Labels ---

# Define la ruta base donde se encuentra el dataset BookCover30.
# ¡ACTUALIZA ESTA RUTA CON LA UBICACIÓN CORRECTA EN TU GOOGLE DRIVE!
BOOKCOVER30_ROOT_PATH = '/content/drive/MyDrive/Tesis/data/raw/book30/'

# Asumimos que la información de las imágenes está en un archivo CSV dentro de BOOKCOVER30_ROOT_PATH.
METADATA_FILE = os.path.join(BOOKCOVER30_ROOT_PATH, 'book30-listing-train.csv')

# Directorio donde se encuentran las imágenes locales.
LOCAL_IMAGES_DIR = os.path.join(BOOKCOVER30_ROOT_PATH, 'title30cat/224x224')

# Cargamos el archivo de metadatos
try:
    metadata_df = pd.read_csv(METADATA_FILE)
    print(f"Metadatos cargados desde: {METADATA_FILE}")

    # Renombrar columnas según lo especificado por el usuario:
    # 'Filename' para la ruta de la imagen, 'Category' para la etiqueta.
    metadata_df = metadata_df.rename(columns={'Filename': 'image_filename', 'Category': 'label'})

    # Construir la ruta absoluta a las imágenes locales
    # Asumimos que 'image_filename' en el CSV es solo el nombre del archivo de imagen.
    metadata_df['absolute_image_path'] = metadata_df['image_filename'].apply(
        lambda x: os.path.join(LOCAL_IMAGES_DIR, x)
    )

    # Filtrar imágenes que realmente existen localmente
    metadata_df = metadata_df[metadata_df['absolute_image_path'].apply(os.path.exists)]
    print(f"Total de imágenes encontradas después de filtrar: {len(metadata_df)}")

    if len(metadata_df) == 0:
        raise FileNotFoundError("No se encontraron imágenes válidas. "
                                  "Verifica el archivo book30-listing-train.csv, la columna 'Filename' "
                                  "y la existencia de los archivos en la carpeta 'title30cat'.")

except FileNotFoundError as e:
    print(f"Error al cargar metadatos o imágenes: {e}")
    print("Asegúrate de que el archivo book30-listing-train.csv existe en la ruta especificada,"
          "que la columna 'Filename' es correcta y que las imágenes existen en 'title30cat'.")
    metadata_df = pd.DataFrame()


if not metadata_df.empty:
    # --- 2.2 Revisar cantidad de clases y distribución ---
    num_classes = metadata_df['label'].nunique()
    print(f"\nNúmero de clases en el dataset: {num_classes}")
    print("Distribución de clases:")
    display(metadata_df['label'].value_counts())

    # --- 2.3 Partición de datos (Train/Validation) ---
    # BookCover30 no trae particiones predefinidas, creamos una simple.

    # Mapear etiquetas a IDs numéricos para PyTorch
    label_to_idx = {label: i for i, label in enumerate(metadata_df['label'].unique())}
    idx_to_label = {i: label for label, i in label_to_idx.items()}
    metadata_df['label_id'] = metadata_df['label'].map(label_to_idx)

    train_df, val_df = train_test_split(metadata_df,
                                        test_size=0.2,
                                        random_state=SEED,
                                        stratify=metadata_df['label_id'])

    print(f"\nImágenes para entrenamiento: {len(train_df)}")
    print(f"Imágenes para validación: {len(val_df)}")

    # --- 2.4 Mostrar ejemplos de portadas por clase ---
    print("\nEjemplos de portadas por clase:")
    fig, axes = plt.subplots(min(num_classes, 5), 5, figsize=(15, min(num_classes, 5) * 3))
    axes = axes.flatten()

    for i, (label_name, group) in enumerate(metadata_df.groupby('label')):
        if i >= len(axes):
            break
        sample_image_path = group.sample(1, random_state=SEED)['absolute_image_path'].iloc[0]
        try:
            img = Image.open(sample_image_path).convert('RGB')
            axes[i].imshow(img)
            axes[i].set_title(label_name, fontsize=8)
            axes[i].axis('off')
        except Exception as e:
            axes[i].set_title(f"Error al cargar: {label_name}", fontsize=8)
            axes[i].axis('off')
            print(f"No se pudo cargar la imagen {sample_image_path}: {e}")

    for j in range(i + 1, len(axes)): # Ocultar ejes restantes
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

### 3. Preparar Dataset y DataLoader

In [ ]:
# --- 3.1 Clase PyTorch Dataset ---
class BookCoverDataset(Dataset):
    def __init__(self, dataframe, preprocess, label_to_idx):
        self.dataframe = dataframe
        self.preprocess = preprocess
        self.label_to_idx = label_to_idx

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        # Cargar ruta de la imagen y etiqueta
        img_path = self.dataframe.iloc[idx]['absolute_image_path']
        label_id = self.dataframe.iloc[idx]['label_id']

        # Cargar imagen con PIL y convertir a RGB
        image = Image.open(img_path).convert("RGB")

        # Aplicar el preprocesamiento oficial de OpenCLIP
        image = self.preprocess(image)

        return image, label_id

# --- 3.2 Obtener Preprocesador de OpenCLIP ---
# Para obtener el preprocesador, necesitamos cargar un modelo de OpenCLIP.
# Por ahora, cargaremos un modelo común para obtener su preprocesador.
# Asegúrate de que este modelo base sea el mismo que usaste para generar los embeddings del catálogo.

# MODEL_NAME = "ViT-B-32" # Ejemplo, el usuario debe especificar el modelo exacto
# PRETRAINED = "openai" # O "laion" u otro que corresponda a tu modelo base

# Define el modelo y el pretrained checkpoint que usaste para el catálogo
# **¡AJUSTA ESTOS VALORES AL MODELO ESPECÍFICO QUE HAS USADO!**
OPENCLIP_MODEL_NAME = "ViT-B-32" # Por ejemplo, 'ViT-L-14', 'ViT-B-32'
OPENCLIP_PRETRAINED_DATASET = "openai" # Por ejemplo, 'openai', 'laion2b_s34b_b79k'

# Crear un modelo dummy solo para obtener la función de preprocesamiento
# El modelo real se cargará y se modificará en la siguiente sección
_, _, preprocess = open_clip.create_model_and_transforms(
    OPENCLIP_MODEL_NAME,
    pretrained=OPENCLIP_PRETRAINED_DATASET,
    device=device # Se carga en el dispositivo para evitar errores de tipo
)

print(f"Preprocesador de OpenCLIP para {OPENCLIP_MODEL_NAME}/{OPENCLIP_PRETRAINED_DATASET} cargado.")

# --- 3.3 Crear instancias de Dataset y DataLoader ---
BATCH_SIZE = 32 # Puedes ajustar este valor según la memoria de tu GPU
NUM_WORKERS = 2 # Número de procesos para cargar datos (ajusta según tu CPU y memoria)

# Datasets
train_dataset = BookCoverDataset(train_df, preprocess, label_to_idx)
val_dataset = BookCoverDataset(val_df, preprocess, label_to_idx)

# DataLoaders
train_dataloader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)
val_dataloader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)

print(f"\nDataset de entrenamiento: {len(train_dataset)} imágenes")
print(f"DataLoader de entrenamiento creado con batch_size={BATCH_SIZE}")
print(f"Dataset de validación: {len(val_dataset)} imágenes")
print(f"DataLoader de validación creado con batch_size={BATCH_SIZE}")

# Verificación rápida de un batch
for images, labels in train_dataloader:
    print(f"\nShape de un batch de imágenes: {images.shape}") # Esperado: [BATCH_SIZE, 3, H, W]
    print(f"Shape de un batch de etiquetas: {labels.shape}") # Esperado: [BATCH_SIZE]
    print(f"Tipos de datos: Imágenes={images.dtype}, Etiquetas={labels.dtype}")
    break

### 4. Cargar OpenCLIP y Estrategia de Adaptación

In [ ]:
# --- 4.1 Cargar el modelo base OpenCLIP ---
# Usamos los mismos parámetros que para obtener el preprocesador en la sección 3.
# Es crucial que este modelo sea el mismo que se usó para generar los embeddings del catálogo original.

print(f"Cargando modelo OpenCLIP: {OPENCLIP_MODEL_NAME} (pretrained: {OPENCLIP_PRETRAINED_DATASET})...")
model, _, _ = open_clip.create_model_and_transforms(
    OPENCLIP_MODEL_NAME,
    pretrained=OPENCLIP_PRETRAINED_DATASET,
    device=device # Aseguramos que el modelo se cargue directamente en el dispositivo correcto
)
model.eval() # Ponemos el modelo en modo evaluación por defecto para inferencia
print("Modelo OpenCLIP cargado y movido a la GPU/CPU.")

# --- 4.2 Estrategia de Adaptación: Fine-tuning del Image Encoder para Clasificación ---
# Como se discutió, la estrategia es simple y robusta: fine-tuning mediante clasificación.
# Esto implica:
# 1. Congelar el Text Encoder (no lo necesitamos para este fine-tuning).
# 2. Congelar parte o la totalidad del Image Encoder pre-entrenado para mantener las características generales.
# 3. Añadir una nueva capa lineal (cabeza de clasificación) sobre el embedding del Image Encoder.
# 4. Entrenar solo esta nueva capa lineal y, opcionalmente, las últimas capas del Image Encoder.

print("\nEstrategia de adaptación seleccionada: Fine-tuning del Image Encoder mediante clasificación.")
print("Se congelará el Text Encoder.")
print("Se añadirá una cabeza de clasificación sobre el Image Encoder y se entrenarán las capas relevantes.")

# Congelar el Text Encoder completamente
for param in model.transformer.parameters():
    param.requires_grad = False

# Congelar el Image Encoder por defecto, y luego lo 'descongelaremos' parcialmente o lo dejaremos congelado
# para entrenar solo la nueva cabeza de clasificación.
# Para la estrategia inicial más simple, congelaremos todo el Image Encoder y solo entrenaremos la nueva cabeza.
# Si se desea un fine-tuning más profundo, se podrían descongelar las últimas capas del Vision Transformer.

# Congelar el Image Encoder (Visión Transformer)
for param in model.visual.parameters():
    param.requires_grad = False

print("Text Encoder y Image Encoder (inicialmente) congelados. Solo se entrenará la nueva cabeza de clasificación.")

# Obtener la dimensión de salida del Image Encoder (tamaño del embedding visual)
# Esto es necesario para la cabeza de clasificación.
with torch.no_grad():
    dummy_input = torch.randn(1, 3, preprocess.transforms[0].size, preprocess.transforms[0].size).to(device)
    embedding_dim = model.encode_image(dummy_input).shape[-1]
print(f"Dimensión del embedding visual del Image Encoder: {embedding_dim}")
print(f"Número de clases para la clasificación: {num_classes}")

### 5. Implementación del Fine-tuning por Clasificación

In [ ]:
import torch.nn as nn

# --- 5.1 Definir el Modelo con Cabeza de Clasificación ---
class OpenCLIPClassifier(nn.Module):
    def __init__(self, openclip_model, embedding_dim, num_classes):
        super().__init__()
        self.openclip_model = openclip_model

        # La estrategia es usar el Image Encoder de OpenCLIP como extractor de características.
        # Ya hemos congelado sus parámetros en la sección anterior para un fine-tuning ligero.

        # Cabeza de clasificación lineal
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, images):
        # Obtener el embedding visual del Image Encoder de OpenCLIP
        # open_clip.model.visual ya aplica la normalización L2 internamente si `normalize_visual_features=True`
        # en create_model_and_transforms (que es el default para muchos modelos).
        # Si no, se debería añadir explícitamente aquí: F.normalize(image_features, dim=-1)
        image_features = self.openclip_model.encode_image(images)

        # Pasar el embedding a la cabeza de clasificación
        logits = self.classifier(image_features)
        return logits

# Instanciar el modelo clasificador
classifier_model = OpenCLIPClassifier(model, embedding_dim, num_classes).to(device)

print(f"\nModelo clasificador creado con Image Encoder de OpenCLIP (congelado) y cabeza lineal de {embedding_dim} a {num_classes} clases.")

# --- 5.2 Estrategia de Congelamiento (Revisión y Ajuste) ---
# Tal como se especificó, los parámetros del Image Encoder (self.openclip_model.visual) están congelados.
# Esto significa que solo los parámetros de self.classifier se actualizarán durante el entrenamiento.

# Verificación de parámetros entrenables
trainable_params = sum(p.numel() for p in classifier_model.parameters() if p.requires_grad)
print(f"Número total de parámetros entrenables: {trainable_params}")

# --- 5.3 Función de Pérdida y Optimizador ---
# Para la clasificación, CrossEntropyLoss es la elección estándar.
# Usaremos AdamW por su buen rendimiento general.

loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=1e-4) # Learning rate inicial, se puede ajustar

print(f"Función de pérdida: {loss_function}")
print(f"Optimizador: {optimizer.__class__.__name__} con learning rate {optimizer.defaults['lr']}")

# --- 5.4 Explicación para regenerar embeddings ---
print("\nDespués del fine-tuning, para regenerar embeddings visuales del catálogo con el modelo adaptado:")
print("Se utilizará `classifier_model.openclip_model.encode_image(image_tensor)` para obtener los embeddings.")
print("La cabeza de clasificación (`classifier_model.classifier`) no se usará para la generación de embeddings, solo para el fine-tuning.")
print("Esto asegura que la salida del encoder visual sea compatible en forma y normalización (si aplica) con el pipeline original.")

### 6. Loop de Entrenamiento y Validación

#### Funciones de Entrenamiento y Validación

Aquí se definen las funciones `train_epoch` y `validate_epoch` para manejar el proceso de una época de entrenamiento y una de validación, respectivamente. Ambas funciones calculan la pérdida y la precisión de la clasificación.

In [ ]:
def train_epoch(model, dataloader, loss_fn, optimizer, device):
    model.train() # Pone el modelo en modo entrenamiento
    total_loss = 0
    correct_predictions = 0
    total_samples = 0

    # Usa tqdm para mostrar una barra de progreso
    for batch_idx, (images, labels) in enumerate(tqdm(dataloader, desc="Training")):
        images = images.to(device) # Mueve las imágenes al dispositivo
        labels = labels.to(device)   # Mueve las etiquetas al dispositivo

        optimizer.zero_grad() # Limpia los gradientes anteriores

        outputs = model(images) # Pasa las imágenes por el modelo
        loss = loss_fn(outputs, labels) # Calcula la pérdida
        loss.backward() # Realiza la retropropagación
        optimizer.step() # Actualiza los pesos del modelo

        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1) # Obtiene las predicciones
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    avg_loss = total_loss / len(dataloader)
    accuracy = correct_predictions / total_samples
    return avg_loss, accuracy

def validate_epoch(model, dataloader, loss_fn, device):
    model.eval() # Pone el modelo en modo evaluación (desactiva dropout, etc.)
    total_loss = 0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad(): # Desactiva el cálculo de gradientes
        for batch_idx, (images, labels) in enumerate(tqdm(dataloader, desc="Validation")):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = loss_fn(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    avg_loss = total_loss / len(dataloader)
    accuracy = correct_predictions / total_samples
    return avg_loss, accuracy


#### Bucle Principal de Entrenamiento

Aquí se orquesta el entrenamiento completo, que incluye:
- Ejecutar múltiples épocas.
- Llamar a las funciones `train_epoch` y `validate_epoch`.
- Implementar un mecanismo de *early stopping* para evitar el sobreajuste.
- Guardar el mejor modelo (basado en la precisión de validación) en Google Drive.

In [ ]:
NUM_EPOCHS = 20 # Número de épocas de entrenamiento (ajusta según sea necesario)
PATIENCE = 5 # Número de épocas sin mejora en la validación antes de early stopping

# Ruta para guardar los checkpoints del modelo
CHECKPOINT_DIR = os.path.join('/content/drive/MyDrive/Tesis/data/models/', 'bookcover30_classifier_checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, 'best_bookcover30_classifier.pth')

best_val_accuracy = -1 # Inicializar con un valor bajo para que cualquier mejora sea registrada
epochs_no_improve = 0

print(f"Iniciando entrenamiento por {NUM_EPOCHS} épocas...")

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
    train_loss, train_accuracy = train_epoch(classifier_model, train_dataloader, loss_function, optimizer, device)
    val_loss, val_accuracy = validate_epoch(classifier_model, val_dataloader, loss_function, device)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}")

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_accuracy)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_accuracy)

    # Early stopping y guardado del mejor modelo
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        epochs_no_improve = 0
        # Guardar solo el state_dict de la cabeza de clasificación y del Image Encoder adaptado si no está congelado
        # En este caso, el Image Encoder está congelado, por lo que solo guardamos la cabeza
        torch.save(classifier_model.state_dict(), BEST_MODEL_PATH)
        print(f"Mejor modelo guardado en {BEST_MODEL_PATH} con una precisión de validación de {best_val_accuracy:.4f}")
    else:
        epochs_no_improve += 1
        print(f"Precisión de validación no mejorada por {epochs_no_improve} épocas. Mejor: {best_val_accuracy:.4f}")
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping activado después de {epoch+1} épocas sin mejora.")
            break

print("\nEntrenamiento finalizado.")

# Opcional: Cargar el mejor modelo después del entrenamiento
classifier_model.load_state_dict(torch.load(BEST_MODEL_PATH))
classifier_model.to(device)
print("Mejor modelo cargado para futuras evaluaciones.")

#### Visualización del Progreso del Entrenamiento

Para entender cómo el modelo aprendió a lo largo de las épocas, podemos visualizar las métricas de pérdida y precisión.

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Accuracy')
plt.plot(history['val_acc'], label='Validation Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### 7. Guardado y Carga del Modelo Adaptado

#### Carga del Mejor Modelo Entrenado

El entrenamiento anterior guardó el mejor modelo (basado en la precisión de validación) en Google Drive. Ahora, vamos a asegurarnos de que podemos cargar este modelo correctamente para su uso futuro.

In [ ]:
# Asegurarse de que el modelo esté en modo evaluación
classifier_model.eval()

print(f"Cargando el mejor modelo desde: {BEST_MODEL_PATH}")
# Cargar el state_dict del mejor modelo
classifier_model.load_state_dict(torch.load(BEST_MODEL_PATH))
classifier_model.to(device)
print("Mejor modelo cargado exitosamente para evaluación.")

### 8. Validaciones y Comparación de Embeddings

Pruebas y verificaciones finales:
- Prueba del modelo adaptado con una imagen individual para generar su embedding.
- Comparación del embedding generado por el modelo base OpenCLIP original versus el modelo adaptado.
- Asegurarse de que los embeddings generados por el modelo adaptado se normalicen a L2, si es el requisito para el sistema de recomendación.
- Confirmar la compatibilidad general del modelo adaptado con el pipeline recomendador, especialmente en términos de formato y dimensionalidad de los embeddings.

#### Generación de Embeddings Visuales con el Modelo Adaptado

Una vez cargado, el modelo (`classifier_model.openclip_model`) se puede utilizar para generar embeddings visuales de nuevas imágenes. Es crucial que el preprocesamiento de estas nuevas imágenes sea idéntico al utilizado durante el entrenamiento.

In [ ]:
def generate_image_embedding(image_path, model, preprocess, device):
    model.eval() # Asegurarse de que el modelo esté en modo evaluación
    image = Image.open(image_path).convert("RGB")
    image_preprocessed = preprocess(image).unsqueeze(0).to(device) # Añadir dimensión de batch

    with torch.no_grad():
        # Obtenemos el embedding del Image Encoder, no de la cabeza de clasificación
        embedding = model.openclip_model.encode_image(image_preprocessed)

        # Asegurar normalización L2 si es necesario
        # OpenCLIP suele normalizar visual_features por defecto. Verificar la configuración.
        # Si no está normalizado, se puede hacer aquí:
        # embedding = torch.nn.functional.normalize(embedding, p=2, dim=-1)
    return embedding.cpu().numpy().flatten()

# Obtener una imagen de ejemplo del conjunto de validación
sample_image_path = val_df.sample(1, random_state=SEED)['absolute_image_path'].iloc[0]
print(f"Generando embedding para la imagen de ejemplo: {sample_image_path}")

adapted_embedding = generate_image_embedding(sample_image_path, classifier_model, preprocess, device)

print(f"Shape del embedding generado por el modelo adaptado: {adapted_embedding.shape}")
print(f"Primeros 5 valores del embedding adaptado: {adapted_embedding[:5]}")

#### Comparación de Embeddings (Modelo Base vs. Modelo Adaptado)

Para verificar la influencia del fine-tuning, podemos comparar los embeddings generados por el modelo OpenCLIP base (sin la cabeza de clasificación y con sus pesos originales) y el modelo adaptado.

In [ ]:
# Cargar el modelo base OpenCLIP sin ninguna modificación
base_model, _, _ = open_clip.create_model_and_transforms(
    OPENCLIP_MODEL_NAME,
    pretrained=OPENCLIP_PRETRAINED_DATASET,
    device=device
)
base_model.eval()

# Generar embedding con el modelo base
# Aquí se usa una instancia temporal de OpenCLIPClassifier para poder llamar a .openclip_model
# Alternativamente, se podría llamar directamente a base_model.encode_image(image_preprocessed)
base_embedding = generate_image_embedding(sample_image_path, OpenCLIPClassifier(base_model, embedding_dim, num_classes), preprocess, device)

print(f"Shape del embedding generado por el modelo base: {base_embedding.shape}")
print(f"Primeros 5 valores del embedding base: {base_embedding[:5]}")

# Calcular la distancia coseno entre los embeddings para ver la diferencia
from sklearn.metrics.pairwise import cosine_similarity

# Reshape para cosine_similarity que espera (n_samples, n_features)
base_embedding_reshaped = base_embedding.reshape(1, -1)
adapted_embedding_reshaped = adapted_embedding.reshape(1, -1)

similarity = cosine_similarity(base_embedding_reshaped, adapted_embedding_reshaped)[0][0]
print(f"\nSimilitud coseno entre el embedding base y el adaptado: {similarity:.4f}")

if similarity < 0.95: # Umbral arbitrario, ajusta según la expectativa
    print("Los embeddings son razonablemente diferentes, lo que sugiere que el fine-tuning ha tenido un impacto.")
else:
    print("Los embeddings son muy similares, el fine-tuning podría no haber alterado significativamente las características (o el dataset era muy pequeño).")

# --- Verificación de normalización L2 ---
# Calcular la norma L2 del embedding adaptado
adapted_embedding_norm = np.linalg.norm(adapted_embedding)
print(f"\nNorma L2 del embedding adaptado: {adapted_embedding_norm:.4f}")

# Comprobar si está cerca de 1 (normalizado)
if np.isclose(adapted_embedding_norm, 1.0):
    print("El embedding adaptado está correctamente normalizado a L2.")
else:
    print("¡Advertencia! El embedding adaptado NO está normalizado a L2. Esto podría afectar los cálculos de similitud.")

---